# ScamShield Continuous Retraining
Triggered by Drive trigger file from the backend. Trains all agent models,
evaluates, and uploads artifacts to R2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install scikit-learn pandas boto3 tldextract -q

In [ ]:
import os, json, time, pickle, re
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

TRIGGER_PATH = '/content/drive/MyDrive/ScamShield/training_data/retrain_trigger.json'
BATCH_PATH   = '/content/drive/MyDrive/ScamShield/training_data/retrain_batch.csv'
ARTIFACTS_DIR = '/content/artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

while not os.path.exists(TRIGGER_PATH):
    time.sleep(30)

with open(TRIGGER_PATH) as f:
    trigger = json.load(f)
print(f'Triggered at {trigger["triggered_at"]} with {trigger["record_count"]} records')

df = pd.read_csv(BATCH_PATH)
print(f'Loaded {len(df)} training records')

In [ ]:
# Helper: extract domain from URL
import tldextract
def get_domain(url):
    try:
        ext = tldextract.extract(url)
        return f'{ext.domain}.{ext.suffix}'.lower() if ext.domain and ext.suffix else ''
    except:
        return ''

def get_vpa_handle(vpa):
    try:
        return vpa.split('@')[0].lower() if '@' in str(vpa) else str(vpa).lower()
    except:
        return ''

version_str = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
print(f'Training version: {version_str}')

In [ ]:
# Agent 1 — Text classifier
print('=== Training Agent 1: Text Classifier ===')
texts = df['text'].fillna('').tolist() if 'text' in df.columns else [''] * len(df)
labels = df['is_scam'].fillna(0).astype(int).tolist() if 'is_scam' in df.columns else [0] * len(df)

vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vec.fit_transform(texts)
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1] if hasattr(clf, 'predict_proba') else y_pred
f1_1 = f1_score(y_test, y_pred)
auc_1 = roc_auc_score(y_test, y_proba) if len(set(y_test)) > 1 else 0.5
print(f'  F1={f1_1:.4f} AUC={auc_1:.4f}')

pickle.dump({'vectorizer': vec, 'classifier': clf},
            open(f'{ARTIFACTS_DIR}/agent_01_v{version_str}.pkl', 'wb'))

In [ ]:
# Agent 2 — URL Classifier
print('=== Training Agent 2: URL Classifier ===')
urls = df['url'].fillna('').tolist() if 'url' in df.columns else [''] * len(df)

url_features = []
for u in urls:
    features = {
        'length': len(u),
        'num_dots': u.count('.'),
        'num_hyphens': u.count('-'),
        'num_slashes': u.count('/'),
        'num_digits': sum(c.isdigit() for c in u),
        'has_https': 1 if 'https' in u else 0,
        'domain_len': len(get_domain(u)),
    }
    url_features.append(features)
X_url = pd.DataFrame(url_features).fillna(0).values
X_ut, X_ute, y_ut, y_ute = train_test_split(X_url, labels, test_size=0.2, random_state=42)

clf_url = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf_url.fit(X_ut, y_ut)

yp_url = clf_url.predict(X_ute)
ypr_url = clf_url.predict_proba(X_ute)[:, 1] if hasattr(clf_url, 'predict_proba') else yp_url
f1_2 = f1_score(y_ute, yp_url)
auc_2 = roc_auc_score(y_ute, ypr_url) if len(set(y_ute)) > 1 else 0.5
print(f'  F1={f1_2:.4f} AUC={auc_2:.4f}')

pickle.dump({'classifier': clf_url},
            open(f'{ARTIFACTS_DIR}/agent_02_v{version_str}.pkl', 'wb'))

In [ ]:
# Agent 7 — Ensemble scorer
print('=== Training Agent 7: Ensemble Scorer ===')
import joblib
# Stack agent 1 & 2 probabilities as features
proba1 = clf.predict_proba(X_train)[:, 1]
proba2 = clf_url.predict_proba(X_ut)[:, 1] if len(X_ut) == len(X_train) else clf_url.predict_proba(X_ut_test)[:, 1]
# Build ensemble features
all_probas = np.column_stack([proba1, proba2]) if 'proba2' in dir() else np.column_stack([proba1, proba1])
# Simplified: use logistic regression over base predictions
from sklearn.linear_model import LogisticRegression
ensemble = LogisticRegression()
ensemble.fit(all_probas[:len(y_train)], y_train[:len(all_probas)])
pickle.dump({'ensemble': ensemble},
            open(f'{ARTIFACTS_DIR}/agent_07_v{version_str}.pkl', 'wb'))
print('  Agent 7 trained (ensemble)')

# Weighted F1 / AUC across all trained agents
weighted_f1 = (f1_1 + f1_2) / 2
weighted_auc = (auc_1 + auc_2) / 2
print(f'=== Weighted F1={weighted_f1:.4f} AUC={weighted_auc:.4f} ===')

In [ ]:
# Write completion signal to Drive
import boto3
from google.colab import userdata

all_artifacts = [f for f in os.listdir(ARTIFACTS_DIR) if f.endswith('.pkl')]

result = {
    'status': 'complete',
    'f1': round(float(weighted_f1), 4),
    'auc': round(float(weighted_auc), 4),
    'artifacts': all_artifacts,
    'version': version_str,
}

os.makedirs('/content/drive/MyDrive/ScamShield/artifacts', exist_ok=True)
with open('/content/drive/MyDrive/ScamShield/artifacts/training_complete.json', 'w') as f:
    json.dump(result, f)
print(f'Completion signal written: F1={result["f1"]} AUC={result["auc"]}')

In [ ]:
# Upload artifacts to R2
R2_ENDPOINT = userdata.get('R2_ENDPOINT_URL')
R2_ACCESS   = userdata.get('R2_ACCESS_KEY')
R2_SECRET   = userdata.get('R2_SECRET_KEY')
R2_BUCKET   = userdata.get('R2_BUCKET_NAME')

if R2_ENDPOINT and R2_ACCESS and R2_SECRET and R2_BUCKET:
    s3 = boto3.client('s3',
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS,
        aws_secret_access_key=R2_SECRET)
    for artifact in all_artifacts:
        s3.upload_file(f'{ARTIFACTS_DIR}/{artifact}',
                       R2_BUCKET, f'models/{artifact}')
        print(f'  Uploaded {artifact}')
else:
    print('R2 credentials not set as Colab secrets — skipping upload')

In [ ]:
# Cleanup trigger file
if os.path.exists(TRIGGER_PATH):
    os.remove(TRIGGER_PATH)
    print('Trigger file cleaned up')
print('Retraining complete')